# Tahap 4 — Deteksi Code Smell dengan UniXcoder (fine-tuning)
**Proyek:** Evaluasi Kinerja LLM dalam Deteksi *Code Smells*

Notebook ini melatih **microsoft/unixcoder-base** untuk mengklasifikasi code smell ke 6 kelas,
menggunakan **stratified 5-fold cross-validation**. Setiap sampel memperoleh prediksi dari model
yang tidak melihatnya saat latih (*out-of-fold*), menghasilkan **463 prediksi** yang langsung
sebanding dengan SonarQube (Tahap 2) dan Qwen (Tahap 3).

Hasil disimpan permanen ke Google Drive sebagai `unixcoder_predictions.csv`.

| Pengaturan | Nilai |
|---|---|
| Model | `microsoft/unixcoder-base` (125 juta parameter) |
| Strategi | Stratified 5-fold CV (seed 42) |
| Panjang token maks | 512 |
| Epoch / fold | 6 |
| Learning rate | 2e-5 |

> **Aktifkan GPU:** Runtime → Change runtime type → **T4 GPU**. UniXcoder ringan; total ~15–40 menit.

## Langkah 1 — Pasang dependensi, cek GPU, kunci seed

In [ ]:
!pip install -q transformers==5.18.0 accelerate==1.15.0
print("Selesai memasang dependensi.")

Selesai memasang dependensi.


In [ ]:
import torch, random, numpy as np, os
from transformers import set_seed
SEED = 42
def kunci_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    torch.cuda.manual_seed_all(s); set_seed(s)
kunci_seed()
print("CUDA tersedia :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU           :", torch.cuda.get_device_name(0))
else:
    print("PERINGATAN: GPU belum aktif. Runtime > Change runtime type > T4 GPU.")

CUDA tersedia : True
GPU           : Tesla T4


## Langkah 2 — Hubungkan Google Drive (penyimpanan permanen)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RESULTS = "/content/drive/MyDrive/code-smell-llm/results/unixcoder"
os.makedirs(RESULTS, exist_ok=True)
print("Hasil akan disimpan permanen di:", RESULTS)
print("Isi folder saat ini:", os.listdir(RESULTS) or "(masih kosong)")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Hasil akan disimpan permanen di: /content/drive/MyDrive/code-smell-llm/results/unixcoder
Isi folder saat ini: ['unixcoder_predictions.csv', 'efficiency', 'token_counts.csv']


## Langkah 3 — Unggah & ekstrak dataset
Sama seperti Tahap 3: kompres folder dataset di komputer Anda
(`Compress-Archive -Path data\dataset -DestinationPath dataset.zip`), lalu unggah `dataset.zip`.

In [ ]:
from google.colab import files
import zipfile, glob
up = files.upload()                       # pilih dataset.zip
zname = next(iter(up))
with zipfile.ZipFile(zname) as z:
    z.extractall("data_in")
gt_paths = glob.glob("data_in/**/ground_truth.csv", recursive=True)
assert gt_paths, "ground_truth.csv tidak ditemukan di dalam zip!"
DATASET_ROOT = os.path.dirname(gt_paths[0])
print("DATASET_ROOT =", DATASET_ROOT)

Saving dataset.zip to dataset (2).zip
DATASET_ROOT = data_in


In [ ]:
import pandas as pd

CLASSES = ["long_method","god_class","feature_envy","data_class","long_parameter_list","non_smells"]
LABEL2ID = {c:i for i,c in enumerate(CLASSES)}
ID2LABEL = {i:c for c,i in LABEL2ID.items()}
NUM_LABELS = len(CLASSES)

gt = pd.read_csv(os.path.join(DATASET_ROOT, "ground_truth.csv"))
gt["id"] = gt["id"].astype(str)

def read_code(row):
    cp = str(row["code_path"]).replace("\\", "/")   # normalisasi path Windows -> Linux
    for p in [os.path.join(DATASET_ROOT, cp),
              os.path.join(DATASET_ROOT, cp.replace("dataset/", "")),
              os.path.join("data_in", cp)]:
        if os.path.exists(p):
            return open(p, encoding="utf-8", errors="replace").read()
    raise FileNotFoundError(cp)

ids       = gt["id"].tolist()
langs     = gt["language"].tolist()
labels_tx = gt["label"].tolist()
codes     = [read_code(r) for _, r in gt.iterrows()]
y         = [LABEL2ID[l] for l in labels_tx]

print("Total sampel:", len(codes))
print(pd.crosstab(gt["label"], gt["language"]))

Total sampel: 463
language             java  python
label                            
data_class             42      24
feature_envy           24      39
god_class              42      40
long_method            42      42
long_parameter_list    42      42
non_smells             42      42


## Langkah 3b — Hitung panjang masukan (truncation 512 token)
Untuk Subbab 3.8 dan Lampiran B tesis. Sel ini **tidak melatih atau memuat model**, hanya tokenizer.
Prasyarat: Langkah 1–3 sudah dijalankan (`codes`, `ids`, `RESULTS` sudah ada). Sel *timing* di bawahnya tidak perlu dijalankan ulang.

In [ ]:
# === PANJANG MASUKAN UNIXCODER: unit > 512 token terpotong saat pelatihan ===
from transformers import AutoTokenizer
from google.colab import files

_tok_cnt = AutoTokenizer.from_pretrained("microsoft/unixcoder-base")   # hanya tokenizer

tc = pd.DataFrame({"id": ids, "language": langs, "label": labels_tx})
# `codes` dibaca dengan read_code() yang sama dengan pelatihan
tc["n_tokens"] = [len(_tok_cnt(c, truncation=False)["input_ids"]) for c in codes]
tc["terpotong_unix"] = tc["n_tokens"] > 512

# Gabungkan dengan prediksi out-of-fold untuk membandingkan recall terpotong vs tidak
pred_path = os.path.join(RESULTS, "unixcoder_predictions.csv")
if os.path.exists(pred_path):
    _p = pd.read_csv(pred_path); _p["id"] = _p["id"].astype(str)
    tc = tc.merge(_p[["id", "unix_pred", "fold"]], on="id", how="left")
    tc["benar"] = tc["unix_pred"] == tc["label"]

out_tc = os.path.join(RESULTS, "token_counts.csv")
tc.to_csv(out_tc, index=False)

print("Unit terpotong (> 512 token) per kelas x bahasa:")
print(tc.groupby(["label", "language"])["terpotong_unix"].agg(["sum", "count"]))
print("\nTotal terpotong: %d dari %d unit" % (tc["terpotong_unix"].sum(), len(tc)))
print("Statistik n_tokens:", tc["n_tokens"].describe().round(0).to_dict())
if "benar" in tc:
    print("\nRecall UniXcoder per kelas, dipisah terpotong / tidak:")
    print(tc.groupby(["label", "terpotong_unix"])["benar"].agg(["mean", "count"]).round(3))
files.download(out_tc)


Unit terpotong (> 512 token) per kelas x bahasa:
                              sum  count
label               language            
data_class          java       14     42
                    python      1     24
feature_envy        java        1     24
                    python      8     39
god_class           java       38     42
                    python     40     40
long_method         java       19     42
                    python     35     42
long_parameter_list java        5     42
                    python     13     42
non_smells          java        7     42
                    python     13     42

Total terpotong: 194 dari 463 unit
Statistik n_tokens: {'count': 463.0, 'mean': 1331.0, 'std': 2750.0, 'min': 18.0, '25%': 173.0, '50%': 393.0, '75%': 1030.0, 'max': 25777.0}

Recall UniXcoder per kelas, dipisah terpotong / tidak:
                                     mean  count
label               terpotong_unix              
data_class          False           0.980     5

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# === TIMING UNIXCODER (efisiensi / RM3) ===
import time, os, pandas as pd, torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from tqdm.auto import tqdm

EFF = os.path.join(RESULTS, "efficiency"); os.makedirs(EFF, exist_ok=True)
_MODEL, _MAXLEN = "microsoft/unixcoder-base", 512
_tk = AutoTokenizer.from_pretrained(_MODEL)
_dev = "cuda" if torch.cuda.is_available() else "cpu"
_mdl = AutoModelForSequenceClassification.from_pretrained(
    _MODEL, num_labels=6).to(_dev).eval()          # head acak: waktu forward identik dgn model terlatih

@torch.no_grad()
def _time_unix(code_text):
    inputs = _tk(code_text, return_tensors="pt", truncation=True,
                 max_length=_MAXLEN).to(_dev)
    torch.cuda.synchronize() if _dev == "cuda" else None
    t0 = time.perf_counter()
    _ = _mdl(**inputs).logits
    torch.cuda.synchronize() if _dev == "cuda" else None
    return (time.perf_counter() - t0) * 1000.0

for _, r in gt.head(3).iterrows():                  # pemanasan
    _time_unix(read_code(r))

rows = []
for _, r in tqdm(gt.iterrows(), total=len(gt), desc="timing UniXcoder"):
    rows.append({"id": str(r["id"]), "language": r["language"],
                 "detector": "UniXcoder", "time_ms": _time_unix(read_code(r))})

df = pd.DataFrame(rows)
df.to_csv(os.path.join(EFF, "unixcoder_timing.csv"), index=False)
print("UniXcoder: %d entri | rata-rata %.1f ms | median %.1f ms"
      % (len(df), df.time_ms.mean(), df.time_ms.median()))

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: microsoft/unixcoder-base
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.bias          | UNEXPECTED | 
pooler.dense.weight        | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


timing UniXcoder:   0%|          | 0/463 [00:00<?, ?it/s]

UniXcoder: 463 entri | rata-rata 26.8 ms | median 25.2 ms


## Langkah 4 — Muat tokenizer & siapkan dataset

In [ ]:
from transformers import AutoTokenizer, DataCollatorWithPadding
MODEL = "microsoft/unixcoder-base"
MAXLEN = 512
tokenizer = AutoTokenizer.from_pretrained(MODEL)
collator = DataCollatorWithPadding(tokenizer)

class CodeDataset(torch.utils.data.Dataset):
    def __init__(self, code_list, label_list):
        self.enc = tokenizer(code_list, truncation=True, max_length=MAXLEN, padding=False)
        self.labels = label_list
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.labels[i])
        return item

print("Tokenizer siap:", MODEL)

Tokenizer siap: microsoft/unixcoder-base


## Langkah 5 — Definisi pelatihan satu fold
Memakai *Trainer* dengan **bobot kelas** (mengimbangi ketidakseimbangan kelas), deterministik
(seed 42). Tiap fold melatih model baru lalu memprediksi fold ujinya.

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from torch import nn
import gc, math, transformers

print("transformers versi:", transformers.__version__)

EPOCHS = 6
BATCH  = 8
LR     = 2e-5

class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, **kw):
        super().__init__(**kw)
        self.class_weights = class_weights
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        w = self.class_weights.to(logits.device) if self.class_weights is not None else None
        loss = nn.CrossEntropyLoss(weight=w)(logits.view(-1, NUM_LABELS), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def train_one_fold(tr_idx, te_idx, fold):
    kunci_seed(SEED + fold)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL, num_labels=NUM_LABELS, id2label=ID2LABEL, label2id=LABEL2ID)
    tr_ds = CodeDataset([codes[i] for i in tr_idx], [y[i] for i in tr_idx])
    te_ds = CodeDataset([codes[i] for i in te_idx], [y[i] for i in te_idx])
    counts = np.bincount([y[i] for i in tr_idx], minlength=NUM_LABELS).astype(float)
    weights = torch.tensor(counts.sum() / (NUM_LABELS * np.maximum(counts, 1)), dtype=torch.float)
    total_steps = math.ceil(len(tr_idx) / BATCH) * EPOCHS
    warmup = math.ceil(0.1 * total_steps)            # setara warmup_ratio=0.1
    args = TrainingArguments(
        output_dir=f"/content/ux_fold{fold}",
        num_train_epochs=EPOCHS, per_device_train_batch_size=BATCH,
        per_device_eval_batch_size=32, learning_rate=LR, weight_decay=0.01,
        warmup_steps=warmup, logging_steps=25, save_strategy="no",
        seed=SEED, report_to="none", fp16=torch.cuda.is_available())
    trainer = WeightedTrainer(class_weights=weights, model=model, args=args,
                              train_dataset=tr_ds, data_collator=collator)
    trainer.train()
    pred = trainer.predict(te_ds).predictions.argmax(-1)
    del model, trainer; gc.collect(); torch.cuda.empty_cache()
    return pred

transformers versi: 5.18.0


## Langkah 5b — Ukur waktu pelatihan satu fold (biaya RM 3)
Melatih ulang **fold 0 saja** untuk mengukur waktunya (± beberapa menit di T4). File prediksi **tidak diubah**.
Prasyarat: Langkah 1–5 sudah dijalankan (termasuk Langkah 4 dan sel definisi `train_one_fold`). Sel *timing* inferensi dan Langkah 6 tidak perlu dijalankan.

In [ ]:
# === WAKTU PELATIHAN FOLD 0 (hanya pengukuran, prediksi lama tidak ditimpa) ===
import time
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

skf_t = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
tr_idx, te_idx = next(iter(skf_t.split(codes, y)))          # fold 0, pembagian sama dengan Langkah 6

if torch.cuda.is_available(): torch.cuda.synchronize()
t0 = time.perf_counter()
preds0 = train_one_fold(tr_idx, te_idx, 0)                   # latih 6 epoch + prediksi fold uji
if torch.cuda.is_available(): torch.cuda.synchronize()
menit = (time.perf_counter() - t0) / 60

f1_0 = f1_score([y[i] for i in te_idx], preds0, labels=list(range(NUM_LABELS)), average="macro")
print(f"Waktu latih + prediksi fold 0 : {menit:.1f} menit (latih {len(tr_idx)}, uji {len(te_idx)} sampel)")
print(f"Estimasi total 5 fold         : {5*menit:.1f} menit")
print(f"Macro-F1 fold 0 (ulang)       : {f1_0:.3f}  <- bandingkan dengan 0,787 di naskah (cek reproducibility)")

with open(os.path.join(RESULTS, "unixcoder_training_time.txt"), "w") as f:
    f.write(f"gpu={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}\n"
            f"fold0_menit={menit:.2f}\nestimasi_5fold_menit={5*menit:.2f}\nfold0_macro_f1_ulang={f1_0:.3f}\n")
print("Tersimpan:", os.path.join(RESULTS, "unixcoder_training_time.txt"))


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: microsoft/unixcoder-base
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.bias          | UNEXPECTED | 
pooler.dense.weight        | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
25,1.759031
50,1.439732
75,0.923300
100,0.670910
125,0.522708
150,0.404233
175,0.257069
200,0.233338
225,0.141391
250,0.143766


Waktu latih + prediksi fold 0 : 1.1 menit (latih 370, uji 93 sampel)
Estimasi total 5 fold         : 5.4 menit
Macro-F1 fold 0 (ulang)       : 0.787  <- bandingkan dengan 0,787 di naskah (cek reproducibility)
Tersimpan: /content/drive/MyDrive/code-smell-llm/results/unixcoder/unixcoder_training_time.txt


## Langkah 6 — Jalankan stratified 5-fold cross-validation

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import time

OUT = os.path.join(RESULTS, "unixcoder_predictions.csv")
prev = pd.read_csv(OUT) if os.path.exists(OUT) else None
done_folds = set(prev["fold"].unique().tolist()) if prev is not None else set()
rows = prev.to_dict("records") if prev is not None else []
print("Fold sudah selesai:", sorted(done_folds) or "(belum ada)")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
fold_f1 = {}
t0 = time.time()
for fold, (tr_idx, te_idx) in enumerate(skf.split(codes, y)):
    if fold in done_folds:
        continue
    print(f"\n========== FOLD {fold} (latih {len(tr_idx)}, uji {len(te_idx)}) ==========")
    preds = train_one_fold(tr_idx, te_idx, fold)
    f1 = f1_score([y[i] for i in te_idx], preds, labels=list(range(NUM_LABELS)), average="macro")
    fold_f1[fold] = f1
    print(f"Fold {fold} Macro-F1: {f1:.3f}")
    for j, i in enumerate(te_idx):
        rows.append({"id": ids[i], "language": langs[i], "true_label": labels_tx[i],
                     "unix_pred": ID2LABEL[int(preds[j])], "fold": fold})
    pd.DataFrame(rows).to_csv(OUT, index=False)
print(f"\nSELESAI dalam {(time.time()-t0)/60:.1f} menit -> {OUT}")
if fold_f1:
    print("Macro-F1 per fold:", {k: round(v,3) for k,v in fold_f1.items()})
    print("Rerata +/- simpangan: %.3f +/- %.3f" % (np.mean(list(fold_f1.values())), np.std(list(fold_f1.values()))))

Fold sudah selesai: [0, 1, 2, 3, 4]

SELESAI dalam 0.0 menit -> /content/drive/MyDrive/code-smell-llm/results/unixcoder/unixcoder_predictions.csv


## Langkah 7 — Metrik akhir (gabungan out-of-fold)

In [ ]:
from sklearn.metrics import f1_score, matthews_corrcoef
pred = pd.read_csv(OUT)
y_true, y_pred = pred["true_label"], pred["unix_pred"]

print("Jumlah prediksi :", len(pred))
print("Akurasi         : %.3f" % (y_true == y_pred).mean())
print("Macro-F1        : %.3f" % f1_score(y_true, y_pred, labels=CLASSES, average="macro"))
print("MCC             : %.3f" % matthews_corrcoef(y_true, y_pred))

cm = pd.crosstab(y_true, y_pred).reindex(index=CLASSES, columns=CLASSES, fill_value=0)
print("\nMatriks kebingungan (baris=asli, kolom=prediksi UniXcoder):")
print(cm.to_string())

print("\nRecall per kelas:")
for c in CLASSES:
    sub = pred[pred["true_label"] == c]
    print(f"  {c:<22}: {(sub['unix_pred']==c).mean():.3f} (n={len(sub)})")

Jumlah prediksi : 463
Akurasi         : 0.812
Macro-F1        : 0.809
MCC             : 0.774

Matriks kebingungan (baris=asli, kolom=prediksi UniXcoder):
unix_pred            long_method  god_class  feature_envy  data_class  long_parameter_list  non_smells
true_label                                                                                            
long_method                   72          0             8           0                    2           2
god_class                      6         56             0           6                    0          14
feature_envy                   6          0            48           0                    3           6
data_class                     0          7             0          58                    0           1
long_parameter_list            0          0             1           0                   83           0
non_smells                     1          8             4          10                    2          59

Recall per kelas:
  

## Langkah 8 — (Opsional) Salin hasil ke komputer

In [ ]:
from google.colab import files
files.download(OUT)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import sys, importlib
print("Python :", sys.version.split()[0])
paket = ["torch", "transformers", "accelerate", "bitsandbytes",
         "sklearn", "scipy", "statsmodels", "pandas", "numpy",
         "matplotlib", "seaborn", "matplotlib_venn"]
for p in paket:
    try:
        m = importlib.import_module(p)
        print(f"{p:<16}: {m.__version__}")
    except Exception as e:
        print(f"{p:<16}: tidak terpasang ({e.__class__.__name__})")
import torch
print("CUDA   :", torch.version.cuda)

Python : 3.13.16
torch           : 2.11.0+cu130
transformers    : 5.18.0
accelerate      : 1.15.0
bitsandbytes    : tidak terpasang (ModuleNotFoundError)
sklearn         : 1.6.1
scipy           : 1.16.3
statsmodels     : 0.15.0
pandas          : 2.2.3
numpy           : 2.1.3
matplotlib      : 3.10.0
seaborn         : 0.13.2
matplotlib_venn : 1.1.2
CUDA   : 13.0
